##Inspect catalogs

In [0]:
%sql
SHOW CATALOGS

catalog
samples
system
workspace


##Inspect schemas

In [0]:
%sql
SHOW SCHEMAS IN workspace

databaseName
bronze
default
gold
information_schema
silver


##Inspect tables

In [0]:
%sql
SHOW TABLES IN workspace.gold

database,tableName,isTemporary
gold,account_balance_analytics,false
gold,account_balance_summary,false
gold,account_balances,false
gold,balance_reconciliation,false
gold,customer_analytics,false
gold,daily_transaction_metrics,false
gold,data_quality_results,false
gold,fraud_alerts,false
gold,pipeline_monitoring,false
gold,streaming_account_risk,false


In [0]:
%sql
DESCRIBE EXTENDED workspace.gold.account_balances

col_name,data_type,comment
account_id,string,null
current_balance,"decimal(18,2)",null
last_updated,timestamp,null
transaction_count,bigint,null
,,
# Delta Statistics Columns,,
Column Names,"account_id, current_balance, last_updated, transaction_count",
Column Selection Method,first-32,
,,
# Detailed Table Information,,


##Demonstrate table grants

In [0]:
%sql
SHOW GRANTS ON TABLE workspace.gold.account_balances

Principal,ActionType,ObjectType,ObjectKey


##Create a restricted view

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.gold.account_balance_analytics AS
SELECT
    account_id,
    current_balance,
    last_updated,
    transaction_count
FROM workspace.gold.account_balances
""")

print("Analyst-safe account balance view created")

Analyst-safe account balance view created


##Verify the view

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.gold.account_balance_analytics
        LIMIT 20
    """)
)

account_id,current_balance,last_updated,transaction_count
ACC000288,-182589.53,2025-12-14T12:18:46.000Z,34
ACC005544,-1065188.31,2025-12-09T07:47:48.000Z,31
ACC005808,1621478.29,2025-12-30T21:56:00.000Z,31
ACC010103,3129362.90,2025-12-31T22:05:09.000Z,36
ACC011629,-871345.91,2025-12-26T00:01:19.000Z,35
ACC014484,-370767.66,2025-12-30T17:18:14.000Z,37
ACC014495,-216496.52,2025-12-22T05:15:44.000Z,36
ACC015230,-71494.68,2025-12-19T07:22:23.000Z,36
ACC020578,-347861.08,2025-12-07T03:02:56.000Z,36
ACC021208,-1957184.59,2025-12-31T20:37:04.000Z,37


##Create masked customer view

In [0]:
spark.sql("""
CREATE OR REPLACE VIEW workspace.gold.customer_analytics AS
SELECT
    customer_id,

    concat(first_name, ' ', last_name) AS customer_name,

    gender,
    city,
    state,
    country,

    customer_segment,
    customer_since

FROM workspace.bronze.customers
""")

print("Customer analytics view created successfully")

Customer analytics view created successfully


##Verify masking

In [0]:
display(
    spark.sql("""
        SELECT *
        FROM workspace.gold.customer_analytics
        LIMIT 20
    """)
)

customer_id,customer_name,gender,city,state,country,customer_segment,customer_since
CUST000001,Sneha Kumar,F,Hyderabad,Telangana,India,REGULAR,2022-06-13
CUST000002,Priya Raju,F,Bengaluru,Karnataka,India,REGULAR,2017-03-29
CUST000003,Rahul Mehta,M,Bengaluru,Karnataka,India,REGULAR,2024-11-06
CUST000004,Vikram Nair,F,Vijayawada,Andhra Pradesh,India,REGULAR,2021-07-11
CUST000005,Ananya Raju,M,Warangal,Telangana,India,PREMIUM,2015-01-15
CUST000006,Aarav Reddy,F,Kolkata,West Bengal,India,VIP,2021-01-25
CUST000007,Priya Gupta,M,Kochi,Kerala,India,REGULAR,2022-06-20
CUST000008,Sneha Sharma,F,Visakhapatnam,Andhra Pradesh,India,VIP,2021-05-02
CUST000009,Vikram Nair,F,New Delhi,Delhi,India,REGULAR,2024-10-08
CUST000010,Sanjay Patel,F,New Delhi,Delhi,India,VIP,2019-01-31


##Check audit history

In [0]:
spark.sql("""
    DESCRIBE HISTORY workspace.silver.transactions
""")

DataFrame[version: bigint, timestamp: timestamp, userId: string, userName: string, operation: string, operationParameters: map<string,string>, job: struct<jobId:string,jobName:string,jobRunId:string,runId:string,jobOwnerId:string,triggerType:string,workspaceId:string>, notebook: struct<notebookId:string>, queryHistoryStatementId: string, clusterId: string, readVersion: bigint, isolationLevel: string, isBlindAppend: boolean, operationMetrics: map<string,string>, userMetadata: string, engineInfo: string]

In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY workspace.gold.account_balances
    """)
)

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-08-22T06:59:04.000Z,75167731114123,kunchesriharshavishnuvardhan@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})",null,List(1964412482424834),3cb29dd6-ed29-4fbc-b056-c8808229fd25,0822-061658-ukbafj4u-v2n,3,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-22T06:53:43.000Z,75167731114123,kunchesriharshavishnuvardhan@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.autoOptimize.optimizeWrite"":""true"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.autoOptimize.autoCompact"":""true""}, statsOnLoad -> true)",null,List(3905651828535934),ad34ea7d-7bb6-452e-9715-2c8f2022bc8d,0822-061658-ukbafj4u-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1698323, numDeletionVectorsRemoved -> 0, numOutputRows -> 135058, numOutputBytes -> 1699700)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-17T09:06:54.000Z,75167731114123,kunchesriharshavishnuvardhan@gmail.com,SET TBLPROPERTIES,"Map(properties -> {""delta.autoOptimize.optimizeWrite"":""true"",""delta.autoOptimize.autoCompact"":""true""})",null,List(1964412482424834),c152a4c2-ea1b-4129-9efa-976092508f3f,0817-074633-b0t0ei79-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-17T07:47:14.000Z,75167731114123,kunchesriharshavishnuvardhan@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3905651828535934),6c6e65a1-190b-478c-9d98-fe90a73d67d6,0817-074633-b0t0ei79-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1698323, numDeletionVectorsRemoved -> 0, numOutputRows -> 135058, numOutputBytes -> 1698323)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-17T06:51:40.000Z,75167731114123,kunchesriharshavishnuvardhan@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3905651828535934),b9a6a4b6-d33d-49bd-9bb0-e35f2912f342,0817-051236-eynzy6z6-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 135058, numOutputBytes -> 1698323)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
